In [4]:
from pathlib import Path
import h5py
import numpy as np

data_dir = Path("../data/raw")

h5_files = list(data_dir.glob("*.h5")) + list(data_dir.glob("*.hdf5"))

print(f"Found {len(h5_files)} HDF5 files:")
for f in h5_files:
    print(f" - {f.name} ({f.stat().st_size / 1e9:.2f} GB)")

Found 1 HDF5 files:
 - train_nominal_000.h5 (0.42 GB)


Tenemos un solo archivo local, de tamaño manejable. Perfecto para exploración y pruebas. No hemos descargado el dataset gigante entero.

In [5]:
path = h5_files[0]
print("Inspecting:", path)

with h5py.File(path, "r") as f:
    print("\nTop-level keys:")
    for key in f.keys():
        obj = f[key]
        if hasattr(obj, "shape"):
            print(f"{key:35s} shape={obj.shape}, dtype={obj.dtype}")
        else:
            print(f"{key:35s} type={type(obj)}")

    print("\nAttributes:")
    for key, value in f.attrs.items():
        print(f"{key}: {value}")

Inspecting: ../data/raw/train_nominal_000.h5

Top-level keys:
EventInfo_mcEventNumber             shape=(100000,), dtype=int32
EventInfo_mcEventWeights            shape=(100000, 27), dtype=float32
fjet_C2                             shape=(100000,), dtype=float32
fjet_D2                             shape=(100000,), dtype=float32
fjet_ECF1                           shape=(100000,), dtype=float32
fjet_ECF2                           shape=(100000,), dtype=float32
fjet_ECF3                           shape=(100000,), dtype=float32
fjet_L2                             shape=(100000,), dtype=float32
fjet_L3                             shape=(100000,), dtype=float32
fjet_Qw                             shape=(100000,), dtype=float32
fjet_Split12                        shape=(100000,), dtype=float32
fjet_Split23                        shape=(100000,), dtype=float32
fjet_Tau1_wta                       shape=(100000,), dtype=float32
fjet_Tau2_wta                       shape=(100000,), dtype=float32

shape=(100000,) significa que el fichero contiene 100,000 jets.
\\\
jet i -> tiene un pT, una label, un weight...

In [6]:
with h5py.File(path, "r") as f:
    labels = f["labels"][:100_000]

print("Labels shape:", labels.shape)
print("Unique labels and counts:", np.unique(labels, return_counts=True))
print("Signal fraction:", labels.mean())

Labels shape: (100000,)
Unique labels and counts: (array([0, 1], dtype=int32), array([49807, 50193]))
Signal fraction: 0.50193


el dataset está prácticamente balanceado.

label = 0 -> background, QCD/light quark/gluon jet
label = 1 -> signal, boosted top jet

Así que la tarea de ML será:
Input: variables del jet
Output: probabilidad de que el jet sea top/signal

In [7]:
with h5py.File(path, "r") as f:
    print("training_weights exists:", "training_weights" in f.keys())
    
    if "training_weights" in f.keys():
        w = f["training_weights"][:100_000]
        print("Weights shape:", w.shape)
        print("min/mean/max:", np.min(w), np.mean(w), np.max(w))

training_weights exists: True
Weights shape: (100000,)
min/mean/max: 0.09697169 0.99864674 3.2382963


In this dataset, each jet must contribute to the training loss with its corresponding `training_weights`, because the simulated signal and background samples can have different or artificial kinematic distributions, especially in jet $p_T$. If we used an unweighted loss, $\mathcal{L}=\frac{1}{N}\sum_{i=1}^N \ell_i$, the network could learn non-physical shortcuts such as associating certain $p_T$ regions with signal/background instead of learning genuine jet substructure. Therefore, we use a weighted loss,
$$
\mathcal{L}_{\mathrm{weighted}}=\frac{1}{N}\sum_{i=1}^N w_i\,\ell_i,
$$
or equivalently sometimes normalized as
$$
\mathcal{L}_{\mathrm{weighted}}=\frac{\sum_{i=1}^N w_i\,\ell_i}{\sum_{i=1}^N w_i},
$$
where $w_i$ is the `training_weights` value for jet $i$. This makes the training objective better reflect the intended physical distribution and is explicitly required for the ATLAS top-tagging open dataset.

el dataset sí contiene jets simulados, no datos “naturales” directos, y además se usan training_weights para reponderar la contribución de cada jet en la loss, de modo que el modelo no aprenda sesgos artificiales de la simulación —por ejemplo diferencias en la distribución de pT, sino la subestructura física del jet.

Aunque todos sean simulados, signal y background no se generan exactamente con el mismo proceso físico ni con la misma distribución de pT: signal viene de boosted top jets y background de jets QCD/light quarks/gluons.
El sesgo artificial sería que la red aprenda: “este rango de pT suele ser signal”, en vez de aprender la forma interna del jet. Por eso se usan training_weights: para equilibrar esas distribuciones y evitar ese atajo.

In [8]:
with h5py.File(path, "r") as f:
    pt = f["fjet_clus_pt"][:10_000]

print("Constituent pt shape:", pt.shape)

mask = pt > 0
multiplicity = mask.sum(axis=1)

print("Multiplicity min/mean/max:", multiplicity.min(), multiplicity.mean(), multiplicity.max())
print("First 10 multiplicities:", multiplicity[:10])

Constituent pt shape: (10000, 200)
Multiplicity min/mean/max: 5 56.1881 200
First 10 multiplicities: [43 36 37 44 98 54 57 63 23 58]


Esto nos dice que tenemos unos 200 constituyentes por jet como maximo. el que menos tiene son 5 constituyentes (y esto justifica enotnces al no ser todos 200 que tengamos que usar zero padding, porque?)

La media son 56 constituyentes. Esto es lo que queriamos, jets con numero variable de particulas

In [9]:
high_level_branches = [
    "fjet_C2", "fjet_D2", "fjet_ECF1", "fjet_ECF2", "fjet_ECF3",
    "fjet_L2", "fjet_L3", "fjet_Qw", "fjet_Split12", "fjet_Split23",
    "fjet_Tau1_wta", "fjet_Tau2_wta", "fjet_Tau3_wta",
    "fjet_Tau4_wta", "fjet_ThrustMaj",
]

with h5py.File(path, "r") as f:
    available_keys = list(f.keys())
    for name in high_level_branches:
        print(name, name in available_keys)

    existing_hl = [name for name in high_level_branches if name in available_keys]
    hl_data = np.stack([f[name][:100_000] for name in existing_hl], axis=1)

print("High-level feature matrix:", hl_data.shape)
print("First 5 feature means:", np.nanmean(hl_data, axis=0)[:5])
print("First 5 feature stds:", np.nanstd(hl_data, axis=0)[:5])

fjet_C2 True
fjet_D2 True
fjet_ECF1 True
fjet_ECF2 True
fjet_ECF3 True
fjet_L2 True
fjet_L3 True
fjet_Qw True
fjet_Split12 True
fjet_Split23 True
fjet_Tau1_wta True
fjet_Tau2_wta True
fjet_Tau3_wta True
fjet_Tau4_wta True
fjet_ThrustMaj True
High-level feature matrix: (100000, 15)
First 5 feature means: [0.12359007 1.912411   1.5516568  0.18077005 0.00750515]
First 5 feature stds: [0.08581127 1.122438   0.8682184  0.26395407 0.03581548]


Una matriz con las cosas mas importantes. Puede ser una muy buena entrada para el MLP inicial. (El README dice explícitamente que hay 15 high-level quantities pensadas como variables resumen del jet)